# Domain-Adaptive Colorization — Pipeline Runner

Runs `main.py` end to end: Grounded-SAM detection/routing → per-domain DDColor experts → soft blending → RGB reconstruction.

**Kaggle settings:** GPU accelerator (T4 x2 or P100) and **Internet ON**.

In [ ]:
# Cell 1: clone the project (branch B)
from pathlib import Path

REPO_DIR = Path("/kaggle/working/domain-adaptive-colorization")
if not REPO_DIR.exists():
    !git clone -b B https://github.com/zaeem886/domain-adaptive-colorization.git {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Cell 2: install all dependencies upfront safely
!pip install -q --no-deps lpips
!pip install -q segment-anything transformers timm einops scikit-image scipy opencv-python-headless huggingface-hub pyyaml tqdm pytest matplotlib pandas

In [ ]:
# Cell 3: run the test suite (no GPU, torch, cv2, or model weights required)
!PYTHONPATH=. python -m pytest -q tests

In [ ]:
# Cell 4: clone DDColor and download the pretrained DDColor-T/L checkpoints
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

DDCOLOR_DIR = Path("/kaggle/working/DDColor")
if not DDCOLOR_DIR.exists():
    !git clone --depth 1 https://github.com/piddnad/DDColor.git {DDCOLOR_DIR}

CHECKPOINT_T = hf_hub_download("piddnad/ddcolor_paper_tiny", "pytorch_model.bin")
CHECKPOINT_L = hf_hub_download("piddnad/ddcolor_paper", "pytorch_model.bin")
CHECKPOINT = CHECKPOINT_T  # used by the single-image demo in Cell 5
print("DDColor:", DDCOLOR_DIR)
print("DDColor-T checkpoint:", CHECKPOINT_T)
print("DDColor-L checkpoint:", CHECKPOINT_L)

In [ ]:
# Cell 5: run the pipeline
IMAGE_PATH = "pics/dog.jpeg"          # change to your own image
OUTPUT_PATH = "outputs/kaggle_colorized.png"

!python main.py \
    --image {IMAGE_PATH} \
    --output {OUTPUT_PATH} \
    --single-checkpoint {CHECKPOINT} \
    --ddcolor-path {DDCOLOR_DIR} \
    --sam-checkpoint sam_vit_b.pth \
    --device cuda \
    --save-domain-map \
    --visualize \
    --stages-dir outputs/stages

In [ ]:
# Cell 6: display the intermediate pipeline stages
import os
from IPython.display import Image as IPImage, display

STAGES_DIR = "outputs/stages"
stages = [
    ("1. Decomposition", "01_decomposition.png"),
    ("2. SAM masks (raw + resolved)", "02_sam_masks.png"),
    ("3. Segments (crop + mask)", "03_segments.png"),
    ("4. Colorized segments", "04_colorized_segments.png"),
    ("5. Blended result", "05_blended.png"),
]
for title, filename in stages:
    path = os.path.join(STAGES_DIR, filename)
    print(title)
    if os.path.exists(path):
        display(IPImage(filename=path))
    else:
        print("  missing:", path)

In [ ]:
# Cell 7: display the input, result, and routed domain map
from IPython.display import Image as IPImage, display

print("Input")
display(IPImage(filename=IMAGE_PATH))
print("Colorized")
display(IPImage(filename=OUTPUT_PATH))
print("Domain map")
display(IPImage(filename=OUTPUT_PATH.replace(".png", "_domains.png")))

### Notes
- `--visualize` saves the intermediate stages to `outputs/stages/`: `01_decomposition.png`, `02_sam_masks.png`, `03_segments.png`, `04_colorized_segments.png`, `05_blended.png`, plus per-region files in `outputs/stages/regions/`.
- To skip detection and colorize the whole image with the general expert, add `--skip-detection` to the command above.
- For real per-domain experts, replace `--single-checkpoint` with `--checkpoints-dir checkpoints/`; the directory should contain `people.pth`, `nature.pth`, `urban.pth`, `animals.pth`, and `fallback.pth`.
- SAM ViT-B (~375 MB) is downloaded automatically on the first run if `sam_vit_b.pth` is missing.

In [ ]:
# Cell 9: Run Person C tests (metrics, plotting, test sets, E3 baseline)
!PYTHONPATH=. python -m pytest -q tests/test_metrics.py tests/test_plotting.py tests/test_test_sets.py tests/test_e3_baseline.py

In [ ]:
# Cell 9b: Validate the fixed COCO IDs and save an audit manifest
from pathlib import Path
from colorize.test_sets import validate_coco_ids, write_coco_metadata_manifest

# Automatically find instances_val2017.json anywhere in /kaggle/input
COCO_ANNOTATIONS = next(Path("/kaggle/input").rglob("instances_val2017.json"), None)

if COCO_ANNOTATIONS is None:
    print('COCO annotations not mounted; ID validation was not run.')
else:
    print(f'Using annotations: {COCO_ANNOTATIONS}')
    validation = write_coco_metadata_manifest(
        COCO_ANNOTATIONS, 'outputs/coco_id_manifest.json'
    )
    print('COCO ID validation report generated at outputs/coco_id_manifest.json')
    for domain, result in validation['domains'].items():
        status = 'PASS' if result['valid'] else 'PARTIAL'
        print(f'  {domain}: {status} ({len(result["mismatches"])} without COCO instance tags)')
    print('Audit manifest saved successfully!')


In [ ]:
# Cell 10: E3 - DDColor-T/L whole-image baselines on domain + mixed-scene sets
from pathlib import Path

from experiments.e3 import E3_SET_NAMES, run_e3_experiment

# Automatically find the COCO val2017 directory anywhere in /kaggle/input.
val2017_dir = next((p for p in Path("/kaggle/input").rglob("val2017") if p.is_dir()), None)
if val2017_dir is None:
    raise FileNotFoundError("COCO val2017 was not found under /kaggle/input")
COCO_ROOT = val2017_dir.parent
print(f'COCO_ROOT: {COCO_ROOT}')
print(f'E3 sets: {list(E3_SET_NAMES)} (100 images expected)')

records = run_e3_experiment(
    coco_root=COCO_ROOT,
    ddcolor_path=str(DDCOLOR_DIR),
    checkpoint_t=CHECKPOINT_T,
    checkpoint_l=CHECKPOINT_L,
    output_csv='outputs/e3_results.csv',
    summary_csv='outputs/e3_summary.csv',
    manifest_csv='outputs/e3_manifest.csv',
    device='cuda',
    gt_root=COCO_ROOT,
    overwrite=True,
)
print(f'E3 complete: {len(records)} records written to outputs/e3_results.csv')
print('Set-level summary: outputs/e3_summary.csv')


In [ ]:
# Cell 11: Plot E3 results
from colorize.results import load_results
from colorize.plotting import plot_metric_bar, plot_metric_comparison
from pathlib import Path

records = load_results("outputs/e3_results.csv")
print(f"Loaded {len(records)} result records")

# Compare both whole-image baselines across the complete 100-image set.
fig = plot_metric_bar(records, metric="colorfulness", title="E3: Colorfulness (T vs L)", output_path="outputs/e3_colorfulness.png")
if fig:
    from IPython.display import Image as IPImage, display
    display(IPImage(filename="outputs/e3_colorfulness.png"))

# If PSNR is available, also plot it
psnr_records = [r for r in records if r.metric == "psnr"]
if psnr_records:
    fig2 = plot_metric_bar(records, metric="psnr", title="E3: PSNR (DDColor-T whole-image)", output_path="outputs/e3_psnr.png")
    if fig2:
        display(IPImage(filename="outputs/e3_psnr.png"))

summary = Path("outputs/e3_summary.csv")
if summary.exists():
    import pandas as pd
    display(pd.read_csv(summary))

### E3 Experiment Notes
- `ddcolor_t_whole` uses DDColor-T (256×256 input, ConvNeXt-T backbone) on the whole image.
- `ddcolor_l_whole` uses DDColor-L (512×512 input, ConvNeXt-L backbone) on the whole image.
- Cell 10 evaluates all 100 fixed images: 20 each from people, nature, urban, animals, and mixed-scene sets.
- `outputs/e3_results.csv` contains per-image metrics; `outputs/e3_summary.csv` contains per-set mean/std values; `outputs/e3_manifest.csv` records the exact files used.
- The E3 runner is also available from the command line as `python experiments/e3.py run`.